# 2. Training data

The translators learn only from data built automatically. No output of a commercial model is used.

| Source | Used by | Needs |
|---|---|---|
| FOLIO training stories | v2, v3, v4 | network |
| DGT legal sentences in six languages, aligned and filtered | v2, v3, v4 | network, a GPU for the alignment check, about 1.5 GB |
| Synthetic stories over ten domains, labels computed by Z3 | v3, v4 | CPU, about 15 minutes |
| ContractNLI clauses, formalized by translator v2 and kept only when Z3 gives the human label | v3, v4 | network, and a GPU for the formalization (notebook 3) |
| Stories over real vocabularies (EuroVoc, FIBO, DPV) in six languages | v4 | CPU, about 20 minutes |

Every step compares what it builds with the file stored in `data/training/`. The steps that query live services
(DGT, EuroVoc, FIBO, DPV) are off by default, because those services change over time; the stored files are the
ones used in the paper.

In [1]:
# Copyright 2026 Javier Marín
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path

REPO = Path.cwd().parent
DATA = REPO / "data"
RUNS = REPO / "runs"      # everything this notebook produces goes here; data/ is never overwritten

import hashlib
import json
import urllib.request
import zipfile

from logicverifier import data, synthetic

TRAINING = DATA / "training"
REBUILD_FROM_LIVE_SOURCES = False   # DGT, EuroVoc, FIBO and DPV


def same(built, name):
    return "identical" if built == data.read_jsonl(TRAINING / name) else "DIFFERENT"

## FOLIO training stories

In [2]:
folio_train = data.read_jsonl(data.download_folio("train", RUNS / "folio" / "folio-train.jsonl"))
len(folio_train)

1004

## DGT sentences in six languages

Sentences of the EU translation memory that exist in English, Spanish, French, German, Dutch and Italian, contain a
modal verb and a negation or a number, and pass alignment checks: the same numbers, a comparable length, no fragment,
and a multilingual embedding (LaBSE) close to the English one. 300 go to the controls of Appendix C and 1,500 to
training.

In [3]:
if REBUILD_FROM_LIVE_SOURCES:
    dgt_test, dgt_train, dgt_report = data.prepare_dgt(RUNS / "dgt")
    print(dgt_report, same(dgt_test, "dgt_test.jsonl"), same(dgt_train, "dgt_train.jsonl"))
else:
    dgt_test, dgt_train = data.read_jsonl(TRAINING / "dgt_test.jsonl"), data.read_jsonl(TRAINING / "dgt_train.jsonl")
    print(json.loads((TRAINING / "dgt_report.json").read_text()))

{'candidates': 132229, 'conflicting_duplicates_dropped': {'es': 14419, 'fr': 3508, 'de': 1080, 'nl': 611, 'it': 659}, 'simple_checks': {'numbers': 44967, 'ok': 78430, 'length': 5013, 'fragment': 3819}, 'embedded': 9000, 'min_sim_threshold': 0.75, 'min_sim_quantiles': {'p05': 0.763, 'p25': 0.849, 'p50': 0.89, 'p75': 0.92}, 'kept_after_embedding': 8649, 'old_test': {'rows': 300, 'fail_simple_checks': 123, 'fail_embedding': 103, 'pass_both': 168}, 'test': 300, 'train': 1500}


## Synthetic stories over ten domains

Formulas first, sentences second, label from Z3. They cover the constructions where translators fail on real text:
either/or, both-or-neither, neither/nor, only, unless, some, no and the scope of quantifiers.

In [4]:
domain_stories = synthetic.domain_stories(n=4000, seed=0)
print(len(domain_stories), "stories", same(domain_stories, "domain_stories.jsonl"))
domain_stories[0]

3270 stories identical


{'domain': 'health',
 'premises': ['Whoever signs a consent form is a pharmacist.',
  'There is no minor that is also a pharmacist.',
  'Elias does not sign a consent form.',
  'Elias is a pharmacist who takes anticoagulants.'],
 'premises-FOL': ['∀x (SignsConsent(x) → Pharmacist(x))',
  '¬∃x (Minor(x) ∧ Pharmacist(x))',
  '¬SignsConsent(elias)',
  'Pharmacist(elias) ∧ TakesAnticoagulants(elias)'],
 'conclusion': 'Elias signs a consent form.',
 'conclusion-FOL': 'SignsConsent(elias)',
 'label': 'False',
 'distractors': ['HasAllergy', 'PrescribesOpioids', 'NeedsReferral']}

## ContractNLI

Real non-disclosure agreements (Koreeda and Manning, 2021). For every Entailment or Contradiction of the training
split, the supporting clauses become the premises and the hypothesis the claim.

In [5]:
archive = RUNS / "contractnli" / "contract-nli.zip"
if not archive.exists():
    archive.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://stanfordnlp.github.io/contract-nli/resources/contract-nli.zip", archive)
with zipfile.ZipFile(archive) as z:
    train_json = json.load(z.open(next(n for n in z.namelist() if n.endswith("train.json"))))
contract_cases, contract_report = data.contractnli_cases(train_json)
print(contract_report, same(contract_cases, "contractnli_cases.jsonl"))

{'not_mentioned': 2820, 'kept': 3115, 'clause_too_long': 887, 'too_many_or_no_clauses': 369} identical


Translator v2 then writes each case in logic, and a translation is kept only when Z3 decides it with the human label.
That step needs a GPU and is in notebook 3; here the stored result is used.

In [6]:
contract_verified = data.read_jsonl(TRAINING / "contractnli_verified.jsonl")
len(contract_verified)

453

## Stories over real vocabularies

Concepts grouped under their parent, from EuroVoc (or Wikidata if EuroVoc does not answer), FIBO and DPV. Predicates
are English names; sentences use the concept's label in each of the six languages, and every story includes true links
of the hierarchy, so that conclusions often need a chain of steps.

In [7]:
if REBUILD_FROM_LIVE_SOURCES:
    groups, groups_report = synthetic.fetch_vocabulary_groups(RUNS / "vocabularies")
    print(groups_report, same(groups, "vocabulary_groups.jsonl"))
else:
    groups = data.read_jsonl(TRAINING / "vocabulary_groups.jsonl")
vocabulary_stories = synthetic.vocabulary_stories(groups, n=6000, seed=1)
print(len(vocabulary_stories), "stories", same(vocabulary_stories, "vocabulary_stories.jsonl"))

3777 stories identical


## Training mixtures

The v2 mixture contains DGT formulas written by the base model, so it is rebuilt in notebook 3 and stored here. The v3
and v4 mixtures are built from the files above and written to `runs/training/`, where notebook 3 reads them.

In [8]:
mixture_v2 = data.read_jsonl(TRAINING / "mixture_v2.jsonl")
mixture_v3, report_v3 = data.build_v3_mixture(domain_stories, contract_verified, folio_train, mixture_v2)
mixture_v4, report_v4 = data.build_v4_mixture(mixture_v3, vocabulary_stories)

# SHA-256 of the mixtures used to train translators v3 and v4
expected = {"v3": "2d82b6d113eb6ae24028a30b711bbfc37b9080e52b957d58ff565b6445c766df", "v4": "96d18b768583194ce321c2cd7a733920e34782a606cedf84ccaf847bb6f255be"}
for name, mixture in (("v3", mixture_v3), ("v4", mixture_v4)):
    path = RUNS / "training" / f"mixture_{name}.jsonl"
    data.write_jsonl(mixture, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{name}: {len(mixture):6d} examples   {'identical' if digest == expected[name] else 'DIFFERENT'}")

v3:   6733 examples   identical
v4:  10510 examples   identical
